# Are transferred labels predictable in the target data?

**Transfer assigns labels; cITCA evaluates their support within the target.** Here two independent synthetic samples share an expression-feature space. A simple source-trained neighbor classifier supplies labels, solely to make the example runnable. This does not benchmark a transfer method.

New to ITCA? [Start with the concept](00_concepts.ipynb).

In [ ]:
from IPython.display import display
from sklearn.neighbors import KNeighborsClassifier
import gauge_sc as gauge

source = gauge.datasets.audit_example(random_state=0)
target = gauge.datasets.audit_example(random_state=1)
source.obs_names = "source_" + source.obs_names
target.obs_names = "target_" + target.obs_names
source_group = source.obs["group"].astype(str)
source_labels = source_group.where(~source_group.str.startswith("Q"), "Q")
# Synthetic expression features are aligned; X is not raw sequencing counts.
assert source.var_names.equals(target.var_names)
transfer = KNeighborsClassifier(n_neighbors=5)
transfer.fit(source.X, source_labels)
target.obs["transferred"] = transfer.predict(target.X)
# Hide simulation assignments: they are not evaluation inputs.
target.obs = target.obs.drop(columns=["group", "simulation_role"])

## Evaluate in the target only

Labels are now fixed. ITCA trains new classifiers inside target folds, using target measurements and target group proportions. It does not score source-to-target prediction accuracy. The name **cITCA** describes this provenance; the API and result column remain `evaluate()` and `itca`.

In [ ]:
result = gauge.evaluate(
    target,
    groupby="transferred",
    views={
        "molecular": {"X": True},
        "qc": {
            "obs": ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
            "log1p": ["total_counts", "n_genes_by_counts"],
        },
    },
    aggregation="fold_mean",
)
display(result.table.round(3))


In [ ]:
result.groups.loc[
    result.groups.view == "molecular",
    ["group", "n_cells", "proportion", "recall", "contribution"],
].round(3)

High target ITCA means the transferred assignments are predictable from the specified target view; it does not establish correct biological label names or validate the entire transfer workflow.

For real data, start with target AnnData containing labels from your chosen transfer tool and a prepared target representation. Feature alignment, unknown labels and any shared-label subset are explicit upstream decisions; this package does not silently filter them. Reversing transfer direction is a different problem. [Transfer recipe](../docs/recipes.md#transferred-labels).

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to assess how predictable the transferred labels are within each target dataset. Inspect feature and label provenance, recommend evaluation settings, and keep the target datasets separate. Explain what support within the target establishes and what it cannot validate.

These tutorial settings are an example; adapt them to your question and data.